# Damaten GitHub GPU Trainer

Google Drive 不要版。Windows が self-play データを push した後にこのノートを実行してください。

**事前に必ず：** `Runtime > Change runtime type > Hardware accelerator > GPU`

In [ ]:
# 作業ディレクトリ: Colab のローカルストレージ（Drive 不要）
from pathlib import Path
WORK_ROOT = Path('/content')
print('WORK_ROOT:', WORK_ROOT)

In [ ]:
import torch
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU が有効になっていません。Runtime > Change runtime type > GPU を確認してください。')

## GitHub Token

Colab の左サイドバー 🔑 **Secrets** に `GITHUB_TOKEN` を登録しておくのが推奨です。  
未登録の場合は直接入力のコメントを外してください。

In [ ]:
import os

# Colab Secrets から取得（推奨）
try:
    from google.colab import userdata
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('Token loaded from Colab Secrets.')
except Exception:
    # Secrets 未登録の場合: 下の行のコメントを外してトークンを貼り付ける
    # os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
    pass

TOKEN = os.environ.get('GITHUB_TOKEN', '')
assert TOKEN, 'GITHUB_TOKEN が設定されていません。上のセルを確認してください。'
print('GITHUB_TOKEN: OK')

In [ ]:
# トークン付き URL でクローン（push 権限あり）
# すでにクローン済みの場合は pull のみ
REPO_DIR = WORK_ROOT / 'Damaten'
TOKEN = os.environ['GITHUB_TOKEN']
CLONE_URL = f'https://{TOKEN}@github.com/Koushien552/Damaten.git'

if not REPO_DIR.exists():
    !git clone "{CLONE_URL}" "{REPO_DIR}"
else:
    %cd "{REPO_DIR}"
    !git remote set-url origin "{CLONE_URL}"
    !git pull --rebase --autostash origin main

%cd "{REPO_DIR}"
!git lfs pull
print('Clone/pull: OK')

In [ ]:
# GPU 学習 & GitHub へ push
!python "{REPO_DIR}/colab/github_gpu_train_and_push.py" \
  --work-root "{WORK_ROOT}" \
  --repo-url https://github.com/Koushien552/Damaten.git \
  --branch main \
  --n 9 \
  --epochs 6 \
  --lr 0.003 \
  --batch-size 2048